# Load WHO (World Health Organization) Indicator Metadata

This notebook fetches health indicator metadata from the WHO Global Health Observatory (GHO) - covering mortality, disease burden, health systems, risk factors, and more.

In [ ]:
%pip install requests tqdm --quiet

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "dev"
TABLE_NAME = "who_indicators"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

FRESH_START = False

In [ ]:
import requests
from tqdm import tqdm
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
schema = StructType([
    StructField("indicator_id", StringType(), False),
    StructField("indicator_name", StringType(), True),
    StructField("long_definition", StringType(), True),
    StructField("source_organization", StringType(), True),
    StructField("source", StringType(), True),
    StructField("topics", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("periodicity", StringType(), True),
    StructField("aggregation_method", StringType(), True),
    StructField("license_type", StringType(), True),
    StructField("embedding_text", StringType(), True),
])

GHO_BASE = "https://ghoapi.azureedge.net/api"

In [ ]:
# Get existing indicator IDs or create table
existing_ids = set()

if FRESH_START:
    spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME}")
    print("Fresh start - dropped existing table")
else:
    try:
        existing_df = spark.sql(f"SELECT indicator_id FROM {FULL_TABLE_NAME}")
        existing_ids = set(row.indicator_id for row in existing_df.collect())
        print(f"Resuming - found {len(existing_ids)} existing records")
    except:
        print("Table doesn't exist yet, starting fresh")

if not existing_ids or FRESH_START:
    empty_df = spark.createDataFrame([], schema)
    empty_df.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite") \
        .saveAsTable(FULL_TABLE_NAME)
    
    # Set table description
    spark.sql(f"""
        COMMENT ON TABLE {FULL_TABLE_NAME} IS 
        'WHO Global Health Observatory indicator metadata covering mortality, disease burden, health systems, risk factors, and global health statistics.'
    """)
    print(f"Created table {FULL_TABLE_NAME} with CDF enabled")

In [ ]:
# Fetch all WHO GHO indicators
print("Fetching WHO GHO indicator list...")
resp = requests.get(f"{GHO_BASE}/Indicator", timeout=60)
resp.raise_for_status()
all_indicators = resp.json().get("value", [])

# Filter out already loaded
indicators = [ind for ind in all_indicators if ind.get("IndicatorCode") not in existing_ids]

print(f"Total WHO indicators: {len(all_indicators)}")
print(f"Already loaded: {len(existing_ids)}")
print(f"Remaining to load: {len(indicators)}")

In [ ]:
# Load indicators
for ind in tqdm(indicators, desc="Loading WHO indicators"):
    try:
        indicator_id = ind.get("IndicatorCode", "")
        indicator_name = ind.get("IndicatorName", "") or ""
        
        # Fetch detailed metadata for this indicator
        long_definition = ""
        try:
            detail_resp = requests.get(f"{GHO_BASE}/Indicator/{indicator_id}", timeout=30)
            if detail_resp.status_code == 200:
                detail = detail_resp.json()
                long_definition = detail.get("value", [{}])[0].get("IndicatorName", "") or ""
        except:
            pass
        
        # Use indicator name as definition if no detailed definition available
        if not long_definition:
            long_definition = indicator_name
        
        # Create embedding text
        embedding_text = f"{indicator_name}. {long_definition}".strip()
        if not embedding_text or embedding_text == ".":
            embedding_text = indicator_name or indicator_id

        record = [(
            indicator_id,
            indicator_name,
            long_definition[:5000] if len(long_definition) > 5000 else long_definition,
            "World Health Organization (WHO)",
            "WHO Global Health Observatory",
            "Health",
            "",
            "",
            "",
            "CC BY-NC-SA 3.0 IGO",
            embedding_text[:5000] if len(embedding_text) > 5000 else embedding_text,
        )]

        row_df = spark.createDataFrame(record, schema)
        row_df.write.format("delta").mode("append").saveAsTable(FULL_TABLE_NAME)

    except Exception as e:
        print(f"Error loading {indicator_id}: {e}")
        continue

print("Done!")

In [ ]:
# Verify
count = spark.sql(f"SELECT COUNT(*) FROM {FULL_TABLE_NAME}").collect()[0][0]
print(f"Total WHO indicators in table: {count}")
display(spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} LIMIT 5"))